# Machine Learning Models

Text was vectorized using TF_IDF with bigrams, capturing two word combinations like "not_good" and "great location". The vocabulary was limited to the top 10,000 features to keep the model lightweight.

SMOTE was applied after vectorization to balance the three classes, generating synthetic samples for the minority classes in the TF-IDF feature space.

In [1]:
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
tqdm.pandas()
from sklearn.feature_extraction.text import TfidfVectorizer
from imblearn.over_sampling import SMOTE

In [2]:
df = pd.read_csv(r"C:\Users\kumri\Downloads\Data Science Notebooks\AirBnB sentiment analysis\sentiment_tags.csv")
df.head()

,date,reviewer_name,comments,language,exclamation_count,question_count,caps_ratio,word_count,cleaned_comments,sentiment
0,1/18/2020,Santiago,Beca its very lovely! 10/10 recommended!,en,2,0,0.025000,6,beca it very lovely! / recommended!,positive
1,8/10/2020,Serena,Renate is very nice and ready to help. The hou...,en,1,0,0.025381,38,renate be very nice and ready to help. the hou...,positive
2,5/21/2022,Rosemary,"Overall the location was excellent, but the co...",en,0,0,0.005682,60,"overall the location be excellent, but the con...",neutral
3,12/11/2019,Ali,"Highly recommended, good value of money a litt...",en,0,0,0.009804,18,"highly recommended, good value of money a litt...",positive
4,8/27/2016,Zhibo,Suze's house is situated in a very nice and qu...,en,0,0,0.022535,59,suze's house be situate in a very nice and qui...,positive


In [3]:
df.shape

(15116, 10)

In [4]:
vectorizer = TfidfVectorizer(max_features=10000, ngram_range=(1,2))
X = vectorizer.fit_transform(df['cleaned_comments'])
y = df['sentiment']

print("X size:", X.shape)
print("y size:", y.shape)

X size: (15116, 10000)
y size: (15116,)


In [5]:
smote = SMOTE(random_state=42)
X_balanced, y_balanced = smote.fit_resample(X,y)
print(pd.Series(y_balanced).value_counts())

sentiment
positive    6801
neutral     6801
negative    6801
Name: count, dtype: int64


In [6]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X_balanced, y_balanced, test_size=0.2,
                                                    random_state=42,
                                                    stratify =y_balanced)

print('Entrenamiento:', X_train.shape)
print('Prueba:',X_test.shape)

Entrenamiento: (16322, 10000)
Prueba: (4081, 10000)


In [7]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train)
y_test_encoded = le.transform(y_test)
print(le.classes_)

['negative' 'neutral' 'positive']


## Results  

Four models were compared using precision, recall, and F1-score
across all three sentiment classes.

In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from xgboost import XGBClassifier
from sklearn.metrics import classification_report

models = {
    'Logisitic Regression': LogisticRegression(random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42),
    'Linear SVC': LinearSVC(random_state=42),
    'XGBoost': XGBClassifier(random_state=42)
}

for name, model in models.items():
    if name == 'XGBoost':
        model.fit(X_train, y_train_encoded)
        y_pred = model.predict(X_test)
        y_pred = le.inverse_transform(y_pred)
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
    
    print(f"\n{name}")
    print(classification_report(y_test, y_pred))



Logisitic Regression
              precision    recall  f1-score   support

    negative       0.89      0.94      0.91      1360
     neutral       0.89      0.83      0.86      1360
    positive       0.92      0.93      0.93      1361

    accuracy                           0.90      4081
   macro avg       0.90      0.90      0.90      4081
weighted avg       0.90      0.90      0.90      4081


Random Forest
              precision    recall  f1-score   support

    negative       0.93      0.95      0.94      1360
     neutral       0.91      0.80      0.85      1360
    positive       0.85      0.93      0.89      1361

    accuracy                           0.89      4081
   macro avg       0.89      0.89      0.89      4081
weighted avg       0.89      0.89      0.89      4081


Linear SVC
              precision    recall  f1-score   support

    negative       0.90      0.97      0.94      1360
     neutral       0.91      0.83      0.87      1360
    positive       0.93   

## Model Selection

**Linear SVC was selected as the final model** based on the highest overall accuracy (91%) and strongest performance detecting negative reviews (97% recall) which is critical for the use case of flagging properties with poor guest experiences.

The vectorizer is saved alongside the model since both are required for inference: new text must be transformed with same vectorizer used during training before being passed to the classifier.


In [9]:
import joblib
joblib.dump(models['Linear SVC'], 
    r'C:\Users\kumri\Downloads\Data Science Notebooks\AirBnB sentiment analysis\models\linear_svc_model1.pkl')
joblib.dump(vectorizer, 
    r'C:\Users\kumri\Downloads\Data Science Notebooks\AirBnB sentiment analysis\models\tfidf_vectorizer1.pkl')

['C:\\Users\\kumri\\Downloads\\Data Science Notebooks\\AirBnB sentiment analysis\\models\\tfidf_vectorizer1.pkl']